# Experiment 2: Optimal Mix Ratio (Heat Map)

**Learning objective:** given a task, a model and a budget, you will analyze the optimal ratio between search and verification and justify it with the mathematical model.

**The budget identity:** total consumption = N x (1+M), where N candidates are each verified M times. With a fixed budget **B = 50**, the legal (N, M) combinations are:

| N | M | Consumption N x (1+M) | Strategy |
|:---:|:---:|:---:|---|
| 50 | 0 | 50 | pure search (majority vote) |
| 25 | 1 | 50 | search + light verification |
| 10 | 4 | 50 | verify-heavy mix |
| 5 | 9 | 50 | pure verification |

The heat map shows accuracy at each (N, M) grid point, so you can see at a glance which mix wins at equal cost.

> **Relation to the literature:** ICLR 2025 (Scaling LLM Test-Time Compute Optimally, arXiv:2408.03314; Figure 7) shows that the ideal sequential-to-parallel ratio changes with difficulty and budget; its Appendix O lists the scanned ratio sets per budget. You test the same phenomenon on the B = 50 heat map. Note the paper's 4x efficiency claim comes from *difficulty-adaptive* allocation (the optimal-ratio idea), not from a fixed-ratio comparison.
>
> **Scope note:** this material measures the **B = 50** budget point only and runs just the four legal combinations; the rest of the grid is in the included cache (`data/cache_subset/`): the full 23-config grid for the three main models (Gemma-3-1B / Gemma-3-4B / Qwen3-4B), and the 6-config portion of the student grid (search N=10/50 plus the four verify splits) for the other three (which covers all four legal combinations). Higher-budget behavior is covered by the decision tree (B >= 100 branch) and by ICLR 2025's budget-scaled results; see Question 4 below.


---

## 1. Configuration (reuse Experiment 1)

Use the same model and the same 6-problem subset as in `01_budget_efficiency.ipynb`, so your results stay comparable to Experiment 1. (To switch models, redo Experiment 1's model selection first.)


In [ ]:
# --- Reuse the configuration from Notebook 01 ---------------------------------
import os, subprocess, sys, json
# Windows: make CUDA runtime DLLs findable (GPU builds of llama-cpp-python need them)
if os.name == "nt":
    _torch_lib = os.path.join(sys.prefix, "Lib", "site-packages", "torch", "lib")
    if os.path.isdir(_torch_lib):
        os.environ["PATH"] = _torch_lib + os.pathsep + os.environ.get("PATH", "")
        os.add_dll_directory(_torch_lib)
# Dataset mirror (huggingface.co unreachable in some regions)
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

# Locate the repository root (walk up until scripts/run_experiment.py is found)
# and switch to it, so relative paths work no matter where Jupyter was started.
_REPO_ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_REPO_ROOT, "scripts", "run_experiment.py")):
    _parent = os.path.dirname(_REPO_ROOT)
    if _parent == _REPO_ROOT:
        raise RuntimeError("Could not locate the repository root (scripts/run_experiment.py not found).")
    _REPO_ROOT = _parent
if os.path.abspath(os.getcwd()) != _REPO_ROOT:
    os.chdir(_REPO_ROOT)
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

MODEL = "gemma-3-1b"      # must match Notebook 01
N_PROBLEMS = 6
OUT_DIR = os.path.join("data", "results", "experiment2_" + MODEL)
# -----------------------------------------------------------------------------
print("Model:", MODEL, "| problems:", N_PROBLEMS)

---

## 2. Run the Four Legal Combinations

Each combination costs exactly 50 calls per problem. Run all four; the results are reusable across the class via the shared 6-problem subset.

**Note:** these four combinations are four of the 8 configs in Notebook 01's preset (N=50/M=0, N=5/M=9, N=10/M=4, N=25/M=1). If your Notebook 01 run covered only its 3-core set (N=50/M=0, N=5/M=9, N=25/M=1), this notebook adds one config (N=10/M=4, 300 calls, about a third of your 3-core run time); with the 2-config fallback it adds two (600 calls). For all three experiment notebooks together, plan for roughly 2-3.5x your Notebook 01 runtime (see the per-model table in `01_budget_efficiency.ipynb`).

> **First run downloads the dataset** (MATH-500, ≈1 MB) from the Hugging Face Hub; later runs reuse the local cache. The setup cell above already tries the mirror automatically; if a download still fails, see the mirror workaround in Notebook 01.


In [ ]:
# Run the 4 legal combinations (incremental records, resumable)
cmd = [sys.executable, "scripts/run_experiment.py",
       "--model", MODEL, "--seeds", "0",
       "--configs", "N=50,M=0", "N=25,M=1", "N=10,M=4", "N=5,M=9",
       "--max-problems", str(N_PROBLEMS),
       "--out-dir", OUT_DIR, "--finalize"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Four combinations finished.")

---

## 3. Build the Heat Map

Run the cells below to plot the heat map from your four runs; missing combinations show up as empty cells.


In [ ]:
# Aggregate the four combinations (tolerant: no data yet -> friendly message)
exp = None
exp_file = os.path.join(OUT_DIR, "exp1.json")
if os.path.isdir(OUT_DIR):
    # Always re-aggregate from records: idempotent and fast, so a rerun of the
    # experiment cell is always reflected here (no stale exp1.json).
    try:
        from scripts.analyze_experiment import main as analyze_main
        analyze_main(["--records", OUT_DIR, "--model", MODEL, "--out", exp_file])
    except Exception as e:
        if not os.path.isdir(OUT_DIR):
            print("Note: no experiment data yet - run the experiment cell first, then re-run this cell.")
        else:
            print("Aggregation failed:", type(e).__name__, str(e)[:200])
if exp_file and os.path.exists(exp_file):
    with open(exp_file, encoding="utf-8") as f:
        exp = json.load(f)
legal = {(50, 0), (25, 1), (10, 4), (5, 9)}
grid = {}
if exp is not None:
    for c in exp["configs"]:
        n, m = c["config"]["N"], c["config"]["M"]
        if (n, m) in legal:
            grid[(n, m)] = c["accuracy"]
    print("Legal combinations:", {str(k): round(v, 3) for k, v in sorted(grid.items())})

In [ ]:
# Plot the heat map (N x M grid, consumption annotated)
import matplotlib.pyplot as plt
import numpy as np

if not grid:
    print("Note: nothing to plot yet - run the experiment cell first.")
else:
    ns = [50, 25, 10, 5]
    ms = [0, 1, 4, 9]
    acc = np.full((len(ns), len(ms)), np.nan)
    for i, n in enumerate(ns):
        for j, m in enumerate(ms):
            acc[i, j] = grid.get((n, m), np.nan)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(acc, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(ms)))
    ax.set_xticklabels(["M=%d" % m for m in ms])
    ax.set_yticks(range(len(ns)))
    ax.set_yticklabels(["N=%d" % n for n in ns])
    for i in range(len(ns)):
        for j in range(len(ms)):
            v = acc[i, j]
            if not np.isnan(v):
                ax.text(j, i, "%.3f\n(consumes %d)" % (v, ns[i] * (1 + ms[j])),
                        ha="center", va="center", fontsize=9)
    ax.set_title("Accuracy Heat Map - " + MODEL + " (B=50, 6 problems)")
    plt.colorbar(im, label="accuracy")
    plt.tight_layout()
    plt.show()

---

## 3b. No-GPU Option: The Cached Heat Map (Cache Layer)

If you did not run the experiment (or your run is still in progress), the included cache (`data/cache_subset/`) contains our results for all six models: the full 23-config grid for the three main models and the 6-config portion of the student grid (search N=10/50 plus the four verify splits) for the other three. Extract your model's four legal combinations below.


In [ ]:
# Cached heat map from the included cache layer (data/cache_subset/)
import json, os

cached = None
curves_path = os.path.join("data", "cache_subset", "curves.json")
if os.path.exists(curves_path):
    for cv in json.load(open(curves_path, encoding="utf-8"))["curves"]:
        if cv.get("key") == MODEL and cv["dataset"] == "MATH-500":
            cached = cv
            break
if cached is None:
    print("Note: no cached curve for", MODEL, "- check data/cache_subset/curves.json")
else:
    legal = {(50, 0), (25, 1), (10, 4), (5, 9)}
    grid = {(c["config"]["N"], c["config"]["M"]): c["accuracy"] for c in cached["configs"]
            if (c["config"]["N"], c["config"]["M"]) in legal}
    print("Cached legal combinations for", MODEL + ":", {str(k): round(v, 3) for k, v in sorted(grid.items())})
    import matplotlib.pyplot as plt
    import numpy as np
    ns = [50, 25, 10, 5]
    ms = [0, 1, 4, 9]
    acc = np.full((len(ns), len(ms)), np.nan)
    for i, n in enumerate(ns):
        for j, m in enumerate(ms):
            acc[i, j] = grid.get((n, m), np.nan)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(acc, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(ms)))
    ax.set_xticklabels(["M=%d" % m for m in ms])
    ax.set_yticks(range(len(ns)))
    ax.set_yticklabels(["N=%d" % n for n in ns])
    for i in range(len(ns)):
        for j in range(len(ms)):
            v = acc[i, j]
            if not np.isnan(v):
                ax.text(j, i, "%.3f\n(consumes %d)" % (v, ns[i] * (1 + ms[j])),
                        ha="center", va="center", fontsize=9)
    ax.set_title("Cached Accuracy Heat Map - " + MODEL + " (B=50, 100 problems)")
    plt.colorbar(im, label="accuracy")
    plt.tight_layout()
    plt.show()

---

## 4. Analysis Questions (Answer in your own words)

1. **Which cell wins?** Is the best accuracy at the pure-search end (N=50/M=0), the pure-verification end (N=5/M=9), or in between? Compare with Experiment 1's curve: do the two agree?

2. **Why does the winner win?** Use the mathematical model of the weak verifier: alpha ≈ 0.62 (fraction of wrong candidates scored >= 7) and beta ≈ 0.37 (fraction of correct candidates scored < 7), both measured from Qwen3-0.6B self-scoring runs (100-problem seed-0, scores on a 1-10 scale). What limits the verify-heavy combinations? Why does generating more candidates (larger N) keep winning?

3. **Verify-heavy vs. search at equal cost:** N=5/M=9 and N=50/M=0 both consume 50 calls. Which one is more accurate in your data? What does this say about "scrutinize fewer candidates" vs. "generate more candidates"?

4. **Does the optimal ratio depend on B?** (This material measures only B = 50.) Reason with the decision tree (three budget tiers: B < 10, 10 ≤ B < 100, B ≥ 100) and with the ICLR 2025 Figure 7 result (the sequential-to-parallel ratio changes with difficulty and budget). Does "search-first at moderate budget" agree with the decision tree's B >= 100 branch, where verification earns its keep only with a strong verifier and a large enough budget? Write your reasoning: this is a *literature + model* question, not a measured one.

<details><summary>Discussion points</summary>

In our runs the winning cell was the search end (N=50/M=0) for all three models at B = 50. The decision tree predicts pure search whenever the verifier is weak, which is consistent with what we measured. At B >= 100 with a strong verifier the tree switches to a mixed ratio; that branch comes from the ICLR 2025 literature, not from measurements here.

</details>

5. **Calculation:** the Chernoff-style bound says the majority-vote failure rate decays roughly as e^(-N*D) for p > 1/2, with D ≈ 0.02 at p = 0.6. For p = 0.6, how large does N need to be for the bound to fall below 5%? (Solve e^(-N*0.02) = 0.05.) Is that N within a B = 50 budget? What does this imply about the *mathematical* guarantee vs. what you measured on your curve?

<details><summary>Discussion points</summary>

N = ln(20) / 0.02 ≈ 150, far beyond the B = 50 budget, and the bound is loose (the true failure rate is well below the bound). The teaching point: the guarantee is exponential but its practical bite at moderate budgets comes from measured behavior, not from the worst-case bound, which is why we run experiments at all.

</details>

6. **Calculation (verifier side):** a weak self-verifier has alpha ≈ 0.62 (fraction of *wrong* candidates scored >= 7) and beta ≈ 0.37 (fraction of *correct* candidates scored < 7). So P(score >= 7 | correct) = 1 − beta ≈ 0.63 and P(score >= 7 | wrong) = alpha ≈ 0.62. In a single comparison between one correct and one wrong candidate, what is roughly the probability that the *wrong* one gets the higher score? What happens to this probability as you re-score M times and average? (This is why M cannot fix a weak verifier: it resamples the same weak signal.)

<details><summary>Discussion points</summary>

With P(high | correct) ≈ 0.63 and P(high | wrong) ≈ 0.62, the two score distributions are nearly identical: the wrong candidate wins roughly half of the *decisive* comparisons, with most of the rest ending in ties (the verifier barely distinguishes). Averaging M independent re-scores of the *same* two candidates narrows the noise around each candidate's mean, but the means themselves are nearly equal, so the win probability stays ≈ 0.5 regardless of M. The verifier's signal quality (alpha/beta), not M, sets the ceiling on what verification can achieve. This is the verifier-side counterpart of the search-side Chernoff calculation: search has a mathematical guarantee to lean on (when p > 1/2), verification has only the verifier's signal quality.

</details>

> **Common misconception alert:** "one inference can be parallelized into many": N candidates are generated *independently*; parallelism changes wall-clock time, not the call budget. The 50 calls are the same whether run serially or in parallel.


---

## Summary

- The heat map shows accuracy over the four legal (N, M) combinations at B = 50.
- Under a weak verifier, the winning cell is expected at the search end, consistent with the decision tree's "pure search when the verifier is weak".
- The B-dependence question is answered with the decision tree and the ICLR 2025 literature, since this material measures B = 50 only.

Proceed to `03_new_task_prediction.ipynb` when ready.
